# llama.cpp
---
[[site]](https://github.com/ggml-org/llama.cpp)

Низеоуровневая библиотека, которую в 2023 написал Болгарский разработчик gergarov для инференса LLAMA моделей. Она понравилась, её начало развивать комьюнити и туда добавили другие модели - главное, чтобы они были в формате GGUF

*"Другие модели" - это некоторый termin abuse, на самом деле можно использовать только те семейства моделей, которые жестко прописали в коде инструмента. На момент 2025 года их около 50 штук, включая llama, mistral mixtral, qween, gemma, phi, falcon, deepseek, rwvk. Полный список архитекутур можно посмотреть [здесь](https://github.com/ggml-org/llama.cpp/blob/master/src/llama-arch.h) Сам же формат GGUF содержит только название семейства, релевантные архитектуре параметы и веса модели. Далее клиент по этому нзаванию выполняет ветку кода соотвествующую модели и делает forward pass.

Под капотом свой собсвтенном движок тензорных вычислений ggml, который появился чуть раньше. Это компактная альтернатива cuBLAS / PyTorch, но поскольку очень узкая (чисто под LLM forward pass), её легко написал один разработчик.

Написана на C++, отсюда название. Физически llama.cpp - это набор утилит, задача которых выполнить forward pass выбранной LLM модели. Основная это клиент llama-cli, а также набор утилит, каждая из которых выполняет какое-то одно действие. Утилиты можно либо установить по ссылке, либо запустить преднастроенный Docker-образ, либо собрать руками из кода (CMake)

Единственный вход llama.cpp = веса модели в формате GGUF и название архитектуры. Вест инференс написан с нуля. Из доступных фишек LLM генерации реализованы KV-Cache, Speculative Decoding, различные режимы сэмплирования

### Примеры использования
Запустить шелл с моделью llama3<br>
`llama-cli -m llama3:1B`

Запустить шелл с моделью llama3<br>
`llama-server -m llama3:1B -host -port`

## Список утилит

#### для генерации
- **llama-cli**<br>
  основная CLI-утилита для генерации текста
- **llama-infill**<br>
  режим fill-in-the-middle генерации (только для моделей с infill-токенами)
- **llama-grammar**<br>
  генерация с ограничениями по формальной грамматике (BNF/GBNF), полезно для структурированного вывода (JSON, DSL)
- **llama-speculative**<br>
  режим speculative decoding с использованием draft-модели для ускорения генерации
- **llama-batched**<br>
  batch-генерация — одновременная для нескольких запросов
- **llama-parallel**<br>
  параллельная генерация в нескольких потоках или независимых контекстах

#### для сервинга
- **llama-server**<br>
  запускает HTTP-сервер с OpenAI-совместимым API (chat, completions, embeddings). Используется для интеграции llama.cpp с внешними приложениями и фреймворками
- **llama-embedding**<br>
  генерирует эмбеддинги для входного текста (sentence/document embeddings)

#### для бенчмаркинга
- **llama-bench**<br>
  бенчмарк инференса модели: измеряет latency, throughput (tokens/sec), влияние числа потоков и GPU/CPU offloading
- **llama-perplexity**<br>
  вычисляет perplexity модели на заданном тексте или датасете. Применяется для оценки качества и сравнения разных квантизаций

#### управление моделями
- **llama-quantize**<br>
  квантизирует модель в форматах GGUF (например, F16 → Q8_0, Q4_K_M). Поддерживает разные схемы и параметры квантизации
- **llama-export-lora**<br>
  слить LoRA-адаптер с базовой моделью
- **llama-copy**<br>
  вспомогательная утилита для копирования и конвертации GGUF-моделей

#### для работы со словарем
- **llama-tokenize**<br>
  преобразует текст во внутренние token IDs модели (для отладки)
- **llama-detokenize**<br>
  преобразует последовательность token IDs обратно в текстовое представление
- **llama-vocab**<br>
  выводит информацию о словаре модели: размер, special tokens

#### для отладки
- **llama-save-load-state**<br>
  сохранить промежуточный state (KV-cache, RNG), потом можно продолжить
- **llama-lookup**<br>
  выводит внутренние lookup-таблицы

## Аргументы команды llama-cli

Это основная команда запускающая инференс, поэтому рассмотрим ее отдельно. Тут много параметров, они все кастомизируют инференс

#### Параметры модели

задается путь к GGUF-модели<br>
-m, --model <path>

если включен режим speculative decoding, можем выбрать draft-модель<br>
--model-draft <path>

подключение LoRA-адаптера к модели<br>
--lora <path>

LoRA с указанием коэффициента масштабирования<br>
--lora-scaled <path>:<scale>

проектор для multimodal моделей (LLaVA и т.п.)
--mmproj <path>


#### Параметры контекста

Размер используемого моделью контекста (N)<br>
-c, --ctx-size <n>

Как работает: по мере генерации накапливается KV-cache, когда доходит до максимального, первые токены начинают стираться в режиме скользящего окна. Рекомендуется ставить равным контексту на трейне (оставить по умолчанию, как указано в файле GGUF), либо больше, но тогда с rope-фактором Когда генерация превышает

тип RoPE scaling - режим масштабирования контекста<br>
--rope-scaling {none,linear,yarn}

коэффициент масштабирования 2x, 4x и т.д.<br>
--rope-scale <float>

базовая частота RoPE<br>
--rope-freq-base <float>

масштаб частоты RoPE<br>
--rope-freq-scale <float>

чтобы первые токены (например, системный промпт) не стирались при превышении маскимального размера контекста, можно их заморозить<br>
--keep <n>


#### Параметры железа

Число CPU-потоков<br>
-t, --threads <n>

Потоки для batch decode<br>
--threads-batch <n>

Количество слоёв, выгружаемых на GPU<br>
--n-gpu-layers <n>

Способ распределения весов между GPU<br>
--split-mode {none,layer,row}

Явное распределение слоёв по нескольким GPU<br>
--tensor-split <f1,f2,...>

Основной GPU<br>
--main-gpu <id>

Отключить memory-mapping модели<br>
--no-mmap

Заблокировать модель в RAM<br>
--mlock

NUMA-aware аллокация<br>
--numa


#### Prompt и ввод

задать первоначальный prompt, с которой начинается chat<br>
`-p, --prompt <text>`

задать первоначальный user prompt из файла<br>
`--prompt-file <path>`

задать system prompt (для chat-моделей)<br>
`--system <text>`

явно задать chat template<br>
`--chat-template <name>`

Задается по имени модели, например `llama3`. Кастомные делать по-моему нельзя<br>Template = тэги и общая структура передаваемого промпта в рамках модели. Важно, чтобы совпадала с теми тегами. на которых модель обучалась

префикс для infill режима<br>
`--in-prefix <text>`

суффикс для infill режима<br>
`--in-suffix <text>`


#### Параметры генерации

Из режима генерации мы выходим когда сгенерировали EOS, превысили порог генерации (входной промпт в этот порог не входит), триггернулась команда "стоп" или Ctrl+D

Максимальное число генерируемых токенов<br>
-n, --n-predict <n>

отключить выход из генерации при получении EOS-токена<br>
--ignore-eos

задать стоп-строку, при генерации которой выходим из генерации, например, ### или <|user|> (можно несколько)<br>
--stop <string>

вручную отредактировать логиты конкретных токенов (вычитаем константу из логита), например, для цензуры<br>
--logit-bias <id:bias>


#### Параметры семплинга

Температура - степень сглаживания софтмакса<br>
--temp <float>

режим Top-K sampling - выбираем из топовых k токенов по вероятности<br>
--top-k <n>

режим Top-P (nucleus) sampling - выбираем из топ p% агрегированной вероятности<br>
--top-p <float>

режим Min-P sampling - выбираем из не менее чем p% от максимальной вероятности<br>
--min-p <float>

режим Typical sampling - выбираем из токенов наиболее близких по surprisal=-logP к общему значению энтропии<br>
--typical <float>

режим Tail Free Sampling - фильтруем токены, начиная с которых вероятность выходит на плато<br>
--tfs <float>

размер скольщящего окна генерации, в котором считаем повторы<br>
--repeat-last-n <n>

штраф за повторение токена - logit / p<br>
--repeat-penalty <float>

Presence penalty - штраф за повторение токена logit - p<br>
--presence-penalty <float>

Frequency penalty - штраф за повторение токена: logit - p * count<br>
--frequency-penalty <float>

использовать Mirostat - адаптивную корректировку температуры<br>
--mirostat <0|1|2>

целевое значение энтропии распределения выходного токена, при превышении уменьшаем, при недоборе увеличиываем<br>
--mirostat-tau <float>

Learning rate в алгоритме Mirostat - сила корректировки температуры<br>
--mirostat-eta <float>


#### Режимы чата

Интерактивный режим (по умолчанию)<br>
-i, --interactive

Сначала запросить ввод пользователя<br>
--interactive-first

Минимальный вывод без форматирования<br>
--simple-io

Цветной вывод<br>
--color

Не печатать prompt<br>
--no-display-prompt

Показать токены prompt’а<br>
--verbose-prompt

Потоковый вывод токенов<br>
--stream


#### Отладка

Seed генератора случайных чисел<br>
`--seed <n>`

Ограничение генерации формальной грамматикой (GBNF)<br>
`--grammar <path>`

JSON-вывод (используется с grammar)<br>
`--json`

Дамп KV-cache (debug)<br>
`--dump-kv-cache`

Отключить логирование<br>
`--log-disable`


#### Speculative decoding

Включить speculative decoding<br>
`--speculative`

Draft-модель<br>
`--draft <path>`

Число токенов, выводимых draft-моделью перед оценкой большой модели<br>
`--draft-tokens <n>`

## SDK
Мржно использовать

# LM Studio
---
[[site]](https://lmstudio.ai/)

LM Studio - это Desktop приложение со своим UI для локального запуска LLM моделей<br>
Приложение пашет поверх рантайма llama.cpp

Локальный запуск LLM моделей требует как минимум 16GB VRAM памяти на видеокарте. Благодаря отгрузке на диск, карта не нужна и достаточно всего 16GB оперативной RAM памяти
Например, веса 1B LLAMA модели занимают всего 1.3GB на диске

Можно подгружать любые модели из репозитория, в том числе из Hugginface

Есть и SDK

Не только запуск, но добавили сервинг (работа в режиме сервера)

LM Studio имплементирует совместимый с OpenAI API = тот же входной и выходной формат, что для ChatGPT<br>Это сделано для универсальности и совместимости приложений. Сейчас по факту стандарт, все движки используют

С 2025 года в LM Studio можно подключать свои MCP-сервера. Для этого достаточно добавить MCP конфиг с


model = lms.llm("qwen/qwen3-4b-2507")
result = model.respond("What is the meaning of life?")

print(result)
```

```
sdfsdf
```

# Ollama
---
[[site]](https://ollama.com)

Ollama - это небольшая утилита (CLI), а также SDK (Python библиотека с клиентом) для локального запуска LLM моделей<br>
Приложение написано поверх рантайма llama.cpp. Разработана в 2023 году, написана на Go

Доступные модели:
- все, которые есть в облачном каталоге ollama<br>https://ollama.com/library<br>
там естественно все модели семейства LLAMA, а также Qwen, Gemma, Mistral и прочее
- любая модель в GGUF формате, но её надо скачать вручную одним из следующих способов
  - вручную скачать из репозитория Huggingface и сконвертировать в GGUF
  - вручную скачать из репозитория Huggingface уже сконвертированную GGUF
  - скачать напрямую через ollama (в FROM можно прописать url модели)
команда ollama

Для подгрузки 7B модели достаточно 8GB RAM

Запустить модель в интекрактивном REPL<br>
`ollama run llama2`

Вывести скачанные модели<br>
`ollama list`

Синтаксис команды<br>
`ollama <command>`
- __serve__<br>запустить основной сервис ollama, он будет отвечать за обработку остальных команд с любых клиентов (cmd, web, UI)
- __pull \<model\>__ <br>скачивает модель из облачного репозитория ollama и размещает в каталоге models
- __run \<model\>__ <br>запускает новый llama.cpp процесс с выбраной моделью и открывает для нее REPL
- __create -f Modelfile__<br>собирает кастомную модель из конфиг файла (база - предобученная модель с параметрами): загружает веса и создает манифест в каталоге моделей models
- __ollama list__<br>показывает все загруженные в каталог модели
- __rm \<model\>__<br>удаляет конкретную модель из каталога
- __signin__<br>войти в акаунт ollama для использования моделей из репозитория
- __signout__<br>выйти из акуанта
- __ps__<br>показать все запущенные модели / llama.cpp процессы - их может быть много, если запустили параллельно<br>при первом запуске ollama run <model> стартует сервисный процесс ollama, через который идет координация всех запускаемых процессов
- __stop \<model\>__<br>дропает процесс llama.cpp
- __show__ --Modelfile<br>показывает параметры модели

__Важно:__ все команды ollama - это тонкий клиент, главный процесс - это центральный HTTP-сервер, который запускается вначале командной ollama serve и отвечает за всю обработку запросов, и от внутренних клиентов (ollama run etc) и от внешних (curl, browser etc). Явно запускать не обязательно, он стартует при первой ollama команде, при поворных к нему просто подсоединяются. Для этого выполняется команда ollama serve. 

Физически модели находятся в каталоге ollama/models Там два типа объектов: manifests = метаданные модели, по которым понимаем, какие модели нам доступны и blobs = физические веса модели

Для кастомизации моделей есть команда ollama create. Конфигурационный файл собирается по аналогии с Docker манифестом. Команды файла модели Modelfile
- FROM<br>какую модель взять за основу (название или url)
    - model
    - model:tag
    - gguf
- PARAMETER<br>переопределение параметров модели (всякие temperature, top-k и прочее)
- SYSTEM<br>скрытые инструкции для модели, передающиеся в промтах ("you are chat bot")
- ADAPTER<br>подключить LoRA адаптер к модели
- MESSAGES<br>история сообщений, передающася в промптах
- TEMPLATE

### API
По умолчнию ollama и так всегда бежит в режиме сервера. В нее можно руками слать OpenAI-совместимые запросы через requests.post() или curl на выделенный порт и ендпоинты типа /api/chat
  в payload прикладываем модель и промты
  stream=True позволяет печатать в real-time

Адрес локального сервера для отправки сообщений<br>
http://localhost:11434/api

Адрес облачного сервера для отправки сообщений (нужно предвариетнльо залогиниться)<br>
https://ollama.com/api

Команды API соотвествуют стандарту ChatGPT и повторяют SDK-шные
- api/generate
- api/chat
- api/embed
- api/show
- api/create
- api/pull
- api/push
- api/delete
- api/copy
- api/tags
- api/ps

В ответе приходят базовые метрики  

### SDK

В целом похоже на LangChain и подобные фреймворки

Сгенерировать ответ
```python
ChatResponse = chat(model, messages)
```

Сгенерировать ответ в режиме streaming (обновление после каждого токена)<br>
`ChatResponse = chat(model, messages, stream=True)`

Под капотом тут вовращается итератор с yeild и поэтому работает асинхронность<br>
`for chunk in response.iter_lines():
    yield parse(chunk)`

Сгенерировать эмбединги для входных токенов
```python
ChatResponse = chat(model, messages)
```

Заставить модель включить режим Thinking
```python 
ChatResponse = chat(model, messages, think=True)
```

Заставить модель генерировать ответ в формате
```python
ChatResponse = chat(model, messages, format='json')
```

Заставить модель генерировать ответ в определенном формате (для описания класса можно использовать pydantic)
```python 
ChatResponse = chat(model, messages, format={})
```

Включить мультимодальность (подключить картинки или PDF-документы к сообщениям)
```python
ChatResponse = chat(model, messages = {'role':'user', 'content':'abc', 'images':<path>}, format='json')
```

Можно использовать свои тулзы
`ChatResponse = chat(model, messages, tool_calls=[func1, func2])`

Пример многошаговой генерации с тулзами
```python 
# первый вызов функции
ChatResponse = chat(model, messages, tool_calls=[])

# результат применения тулза
tool_result = F(tool_calls[0])

messages.append({'role':'tool', 'conent':''})

# повотрно генерируем 
ChatResponse = chat(model, messages, tool_calls=[])
```

Можно использовать Web-поиск, который возвращает списко ссылок на документы<br>
`response = ollama.web_search(query)`

#### Message structure
Структура сообщений тут соотвествует стандарту OpenAI
- role
  - user
  - assistant - ответ модели
  - system - инструкция для модели
- contents - текст сообщения
- images - картинки

### Web-клиенты
У ollama нет визуального интерфейса, но существуют внешние инструменты, такие как [[Openwebui]](https://openwebui.com/) и [[Msty.app]](http://msty.app/), которые можно настроить - будет UI обертка для взаимодействия с LLM моделями по типу ChatGPT

#### RAG
Функционала в ollama нет. Но можно либо собрать пайплайн с внешними векторной базы или прибегнуть к более верхнеуровнему фреймворку, который умеет подключать ollama

### Docker
Можно запустить предварительно собраный Docker образ с ollama сервером<br>
`docker run -d -v ollama:/root/.ollama -p 11434:11434 --name ollama ollama/ollama`

и работать с ним по API или настроить клиент ollama.Client() на любой машине






# vLLM
---
[[site]](https://github.com/vllm-project/vllm)

Инструмент сервинга, разрботанный в Berkley в 2023 году. Используется как production версия для крупных проектах<br>
Алгоритмическая новация - это PagedAttention

Как в vLLM работает serving:
- запускается API сервер, который принимает запросы совместимые с OpenAI стилизованном API
- использует HF TRansformers для загрузки модели
- подгружает свои CUDA ядра для PagedAttention
- параллельно с HTTP-сервером запускается Scheduler
    - ставит запросы в очередь
    - создает под них таски
    - проверяет статус выполнения
    - дергает ядра генерации
- если пользователь просил stream=True, то после каждой итерации отправляется новый токен 

Почему кастомные CUDA ядра?<br>
PyTorch ядра неоптимизированы под конкретную задачу. Утверждается, что можно ускорить обработку 1.5-3x

### PagedAttetnnion
[[paper]](https://arxiv.org/pdf/2309.06180)

__Идея:__ делить KV-Cache на части

## GGUF
Формат хранения весов модели. Стал стандартном для класса инференс-движков llama.cpp / ollama / LM-Studio GGUF = "GGML universal format"

<img src="img/gguf1.png" width=550>

<img src="img/gguf.png" width=300>

Что содержит GGUF файл:
- словарь токенизатора
- веса слоев Attetnion (K, V, W, O), MLP (W1, W2)
- метаинформация (архитектура модели, параметры квантования

Конвертинг<br>
`python convert-hf-to-gguf.py model_dir --outtype q4_k_m`



# LM Studio

```python
import lmstudio as lms

model = lms.llm("qwen/qwen3-4b-2507")
result = model.respond("What is the meaning of life?")

print(result)
```

```
sdfsdf
```

# vLLM

## пример 1

In [ ]:
from vllm import LLM, SamplingParams

# 1. Создаём параметры сэмплинга
sampling_params = SamplingParams(
    temperature=0.7,
    top_p=0.9,
    max_tokens=64,
)

# 2. Загружаем модель (любую HF)
llm = LLM(model="meta-llama/Llama-3-8B-Instruct")

# 3. Прогоняем запросы (можно список)
prompts = [
    "Explain the concept of Mixture-of-Experts in simple terms.",
    "Write a short haiku about GPUs.",
]

outputs = llm.generate(prompts, sampling_params)

# 4. Разбираем ответы
for i, out in enumerate(outputs):
    prompt = prompts[i]
    generated_text = out.outputs[0].text  # берём 0-й вариант
    print("PROMPT:", prompt)
    print("RESPONSE:", generated_text)
    print("=" * 80)


In [ ]:
Можно стримить ответ

In [ ]:
from vllm import LLM, SamplingParams

llm = LLM(model="meta-llama/Llama-3-8B-Instruct")
sampling_params = SamplingParams(temperature=0.7, max_tokens=50)

prompt = "List three advantages of using vLLM for LLM serving."

# streaming=True вернёт итератор по токенам
for output in llm.generate(prompt, sampling_params, streaming=True):
    # output.outputs[0].text содержит накопленный текст
    print(output.outputs[0].text, end="", flush=True)

print()


В vLLM можно запускать в режиме сервера странным способом - через выполнение Python модуля<br>


| Модуль                                   | Назначение                      |
|-------------------------------------------|----------------------------------|
| `vllm.entrypoints.openai.api_server`      | OpenAI-совместимый API сервер    |
| `vllm.entrypoints.api_server`             | Нативный vLLM API сервер         |
| `vllm.entrypoints.llm`                    | CLI генератор                    |
| `vllm.entrypoints.chat`                   | Чат в терминале                  |
| `vllm.entrypoints.tokenizer`              | Отладка токенизатора             |
| `vllm.entrypoints.convert_lora_weights`   | Конвертация LoRA                 |
| `vllm.entrypoints.slurm.*`                | Запуск в SLURM                   |
| `vllm.entrypoints.megablocks`             | Конверсия в megablocks           |


In [ ]:
python -m vllm.entrypoints.openai.api_server \
  --model meta-llama/Llama-3-8B-Instruct \
  --host 0.0.0.0 \
  --port 8000


In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="EMPTY"  # vLLM игнорирует, но поле нужно
)

response = client.chat.completions.create(
    model="meta-llama/Llama-3-8B-Instruct",
    messages=[
        {"role": "user", "content": "Explain what vLLM is in 3 bullet points."}
    ],
    max_tokens=100,
    temperature=0.7,
)

print(response.choices[0].message.content)


In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="EMPTY",
)

stream = client.chat.completions.create(
    model="meta-llama/Llama-3-8B-Instruct",
    messages=[{"role": "user", "content": "Give me a bullet list of 5 serving engines."}],
    max_tokens=128,
    stream=True,
)

for chunk in stream:
    delta = chunk.choices[0].delta.content or ""
    print(delta, end="", flush=True)

print()


In [ ]:
import requests
import json

url = "http://localhost:8000/v1/chat/completions"

payload = {
    "model": "meta-llama/Llama-3-8B-Instruct",
    "messages": [
        {"role": "user", "content": "Summarize the concept of PagedAttention."}
    ],
    "max_tokens": 100,
    "temperature": 0.7,
}

headers = {
    "Content-Type": "application/json",
    "Authorization": "Bearer EMPTY"
}

resp = requests.post(url, data=json.dumps(payload), headers=headers)
print(resp.json()["choices"][0]["message"]["content"])


In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="EMPTY",
)

resp = client.embeddings.create(
    model="some-embedding-model-or-llm",
    input=[
        "First sentence to embed.",
        "Second sentence to embed.",
    ],
)

for i, emb in enumerate(resp.data):
    print(f"Vector {i} length:", len(emb.embedding))


In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="EMPTY",
)

resp = client.chat.completions.create(
    model="meta-llama/Llama-3-8B-Instruct",
    messages=[{"role": "user", "content": "Give me 3 random animal names."}],
    max_tokens=32,
    temperature=1.2,   # более "безумный"
    top_p=0.85,
    presence_penalty=0.0,
    frequency_penalty=0.2,
)

print(resp.choices[0].message.content)


In [ ]:
# Модель 1
python -m vllm.entrypoints.openai.api_server \
  --model meta-llama/Llama-3-8B-Instruct \
  --port 8000 &

# Модель 2
python -m vllm.entrypoints.openai.api_server \
  --model mistralai/Mistral-7B-Instruct-v0.2 \
  --port 8001 &

llama_client = OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")
mistral_client = OpenAI(base_url="http://localhost:8001/v1", api_key="EMPTY")


In [ ]:
from fastapi import FastAPI
from vllm import LLM, SamplingParams

app = FastAPI()
llm = LLM(model="meta-llama/Llama-3-8B-Instruct")

@app.post("/generate")
async def generate(payload: dict):
    prompt = payload["prompt"]
    params = SamplingParams(
        temperature=payload.get("temperature", 0.7),
        max_tokens=payload.get("max_tokens", 64),
    )
    outputs = llm.generate(prompt, params)
    return {"text": outputs[0].outputs[0].text}


In [ ]:
from vllm import LLM, SamplingParams

llm = LLM(model="meta-llama/Llama-3-8B-Instruct")

sampling_params = SamplingParams(
    n=3,               # 3 варианта ответа
    temperature=0.9,
    top_k=40,
    top_p=0.9,
    max_tokens=64,
    stop=["\n\n"],     # остановка по токенам/строкам
)

prompt = "Give me 3 short taglines for a serving engine library."

outputs = llm.generate(prompt, sampling_params)

for i, o in enumerate(outputs[0].outputs):
    print(f"=== Variant {i+1} ===")
    print(o.text)
